In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from category_encoders import TargetEncoder
import warnings

warnings.filterwarnings('ignore')

print("\n--- 🧩 VERİ ÇAPRAZLAMA (FEATURE CROSSING) & MİKRO-SEGMENTASYON ---")

# 1. VERİLERİ YÜKLÜYORUZ
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url)
orig = orig.rename(columns={'customerID': 'id'}).set_index('id')

y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)
y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])
df_all = pd.concat([X_train_full, test], keys=['train', 'test'])

df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce')
df_all.loc[df_all['tenure'] == 0, 'TotalCharges'] = df_all['MonthlyCharges']
df_all['TotalCharges'].fillna(0, inplace=True)

# -------------------------------------------------------------------
# 2. ÖZNİTELİK ÇAPRAZLAMA (Süper Kategoriler Üretme)
# -------------------------------------------------------------------
print("Mikro-Segmentler (Süper Kategoriler) Oluşturuluyor...")
# Profil 1: Ödeme ve Sözleşme Profili
df_all['Micro_Segment_Billing'] = df_all['Contract'] + "_" + df_all['PaymentMethod']
# Profil 2: İnternet ve Risk Profili (Güvenlik ve Destek almayan Fiber kullanıcıları çok risklidir)
df_all['Micro_Segment_Tech'] = df_all['InternetService'] + "_" + df_all['OnlineSecurity'] + "_" + df_all['TechSupport']
# Profil 3: Sosyal Profil
df_all['Micro_Segment_Social'] = df_all['gender'] + "_" + df_all['Partner'] + "_" + df_all['Dependents']

# -------------------------------------------------------------------
# 3. İSTATİSTİKSEL SAPMA (Z-SCORE) MÜHENDİSLİĞİ
# -------------------------------------------------------------------
# Müşteri, kendi "Ödeme ve Sözleşme" grubundakilere göre ne kadar fazla/az ödüyor?
group_mean = df_all.groupby('Micro_Segment_Billing')['MonthlyCharges'].transform('mean')
group_std = df_all.groupby('Micro_Segment_Billing')['MonthlyCharges'].transform('std').fillna(1)
df_all['Z_Score_MonthlyCharges'] = (df_all['MonthlyCharges'] - group_mean) / group_std

# Süre (Tenure) açısından kendi teknoloji grubuna göre durumu
group_tenure_mean = df_all.groupby('Micro_Segment_Tech')['tenure'].transform('mean')
df_all['Diff_Tenure_From_Tech_Group'] = df_all['tenure'] - group_tenure_mean

# Bildiğimiz en iyi sihirli özellikleri unutmuyoruz
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']

# Tüm kategorik değişkenleri seç (Yeni ürettiğimiz Süper Kategoriler dahil)
cat_cols = df_all.select_dtypes(include=['object']).columns.tolist()

X_train_final = df_all.xs('train')
X_test_final = df_all.xs('test')

print(f"Toplam özellik sayısı: {X_train_final.shape[1]}")
print("Çaprazlanmış özelliklere K-Fold Target Encoding uygulanarak model eğitiliyor...\n")

# -------------------------------------------------------------------
# 4. XGBOOST EĞİTİMİ (Target Encoding ile Sızıntısız)
# -------------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx].copy(), y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx].copy(), y_train_full.iloc[valid_idx]
    X_te_test = X_test_final.copy()
    
    # Süper kategorileri sayısallaştırmak için Target Encoder kullanıyoruz (One-Hot yaparsak on binlerce sütun olur!)
    encoder = TargetEncoder(cols=cat_cols, smoothing=10)
    
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_va[cat_cols] = encoder.transform(X_va[cat_cols])
    X_te_test[cat_cols] = encoder.transform(X_te_test[cat_cols])
    
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_te_test)[:, 1] / skf.n_splits
    
    print(f"Feature Crossing Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 Çapraz Veri Mühendisliği OOF Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_feature_crossing.csv", index=False)
print("✅ Mikro-Segmentasyon gönderim dosyası kaydedildi!")


--- 🧩 VERİ ÇAPRAZLAMA (FEATURE CROSSING) & MİKRO-SEGMENTASYON ---
Mikro-Segmentler (Süper Kategoriler) Oluşturuluyor...
Toplam özellik sayısı: 27
Çaprazlanmış özelliklere K-Fold Target Encoding uygulanarak model eğitiliyor...

Feature Crossing Fold 1 AUC: 0.9159
Feature Crossing Fold 2 AUC: 0.9163
Feature Crossing Fold 3 AUC: 0.9140
Feature Crossing Fold 4 AUC: 0.9151
Feature Crossing Fold 5 AUC: 0.9162

🏆 Çapraz Veri Mühendisliği OOF Skoru: 0.9155
✅ Mikro-Segmentasyon gönderim dosyası kaydedildi!
